In [21]:
# import os
# import osmnx as ox
# from tqdm import tqdm


# def get_data(place, tags, filename):
#     # generate output filename
#     csvfilename = os.path.join("data", filename + ".csv")
#     shpfilename = os.path.join("data", filename + ".shp")
    
#     # Get all convenience in Italy
#     gdf = ox.features.features_from_place(place, tags)

#     # Print the number of convenience found
#     print(f"Found {len(gdf)} records in {place}")

#     # Save to CSV
#     gdf.to_csv(csvfilename, index=True)
#     print(f"Saved csv data to {csvfilename}")

#     # Save location data
#     gdf_points = gdf[gdf.geometry.geom_type == 'Point']

#     # Save to Shapefile
#     gdf_points.to_file(shpfilename)
#     print(f"Saved shp data to {shpfilename}")

In [22]:
# city_names = [
#     'Cavallino-Treporti, Veneto, Italy', 'Venezia, Veneto, Italy', 'San Donà di Piave, Veneto, Italy',
#     'Campagna Lupia, Veneto, Italy', 'Dolo, Veneto, Italy', 'Jesolo, Veneto, Italy',
#     'Chioggia, Veneto, Italy', 'Musile di Piave, Veneto, Italy', 'Marcon, Veneto, Italy',
#     "Quarto d'Altino, Veneto, Italy", 'Mira, Veneto, Italy'
# ]

# # TODO: 
# tags = {
#     'shop': True
# }

# for city in tqdm(city_names):
#     filename = '_'.join(list(tags.keys())) + '_' + city.replace(",", "").replace(" ", "_").replace("'", "").replace("-", "")
#     get_data(city, tags, filename)

In [23]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
from tqdm import tqdm


def get_city_data(places, tags, dir, filename):
    # try:
    #     gdf = ox.features.features_from_place(place, tags)
    #     gdf_points = gdf[gdf.geometry.geom_type == 'Point']
    #     gdf_points["city"] = place  # Add city name for reference
    #     print(f"Found {len(gdf_points)} point records in {place}")
    #     return gdf_points
    # except Exception as e:
    #     print(f"Error fetching data for {place}: {e}")
    #     return gpd.GeoDataFrame(columns=["geometry"])  # Return empty GeoDataFrame with geometry column
    # def get_data(place, tags, filename):
    
    data = []
    data_points = []


    for place in tqdm(places, desc="Fetching OSM data"):
        try:
            # Get data
            gdf = ox.features.features_from_place(place, tags)
            gdf["city"] = place  # Add city name for reference
            # gdf_points = gdf[gdf.geometry.geom_type == 'Point']  # Only keep point geometries

            gdf_points = gdf
            gdf_points["city"] = place  # Add city name for reference
            for k in tags.keys():
                gdf_points[k] = gdf.loc[:, k]
            print(f"Found {len(gdf_points)} point records in {place}")
            data.append(gdf)
            data_points.append(gdf_points)
        except Exception as e:
            print(f"Error fetching data for {place}: {e}")
            continue  # Skip this place if error occurs


    # merge all data
    if len(data):
        final_gdf = gpd.GeoDataFrame(pd.concat(data, ignore_index=True), crs=data[0].crs)
        final_gdf_points = gpd.GeoDataFrame(pd.concat(data_points, ignore_index=True), crs=data[0].crs)
            # get output file name
        if os.path.exists(dir):
            csvfilename = os.path.join(dir, filename + '.csv')
            shpfilename = os.path.join(dir, filename + ".shp")

            # Save to CSV
            final_gdf.to_csv(csvfilename, index=True)
            print(f"Saved csv data to {csvfilename}")

            # Save to Shapefile
            final_gdf_points.to_file(shpfilename)
            print(f"Saved shp data to {shpfilename}")

        else:
            print(f"路径{dir}不存在，结束程序")
            return
    else:
        print("No data was collected.")

In [24]:
# City list
city_names = [
    'Cavallino-Treporti, Veneto, Italy', 'Venezia, Veneto, Italy', 'San Donà di Piave, Veneto, Italy',
    'Campagna Lupia, Veneto, Italy', 'Dolo, Veneto, Italy', 'Jesolo, Veneto, Italy',
    'Chioggia, Veneto, Italy', 'Musile di Piave, Veneto, Italy', 'Marcon, Veneto, Italy',
    "Quarto d'Altino, Veneto, Italy", 'Mira, Veneto, Italy'
]

# 修改这里获取不同类型的数据
# 参考 https://wiki.openstreetmap.org/wiki/Map_features#Building 获取数据类型定义
# 参考代码 https://github.com/gboeing/osmnx-examples/blob/22426e0a6c5040cc155f9cc5202e7212b5cc1818/notebooks/16-download-osm-geospatial-features.ipynb
# Tags to search for
# tags = {
#     'shop': True 
# }

tags = {
    "landuse": ["residential", "commercial", "industrial"]
}


# Ensure output directory exists
os.makedirs("data", exist_ok=True)

# 修改输出文件夹和文件名
outputdir = r'G:\CODE\data\05_landuse_mix'
filename = 'landuse_data'

get_city_data(city_names, tags, outputdir, filename)

Fetching OSM data:   9%|▉         | 1/11 [00:02<00:21,  2.19s/it]

Found 22 point records in Cavallino-Treporti, Veneto, Italy


Fetching OSM data:  18%|█▊        | 2/11 [00:07<00:37,  4.21s/it]

Found 201 point records in Venezia, Veneto, Italy


Fetching OSM data:  27%|██▋       | 3/11 [00:09<00:24,  3.12s/it]

...

Found 4 point records in Quarto d'Altino, Veneto, Italy


Fetching OSM data: 100%|██████████| 11/11 [00:56<00:00,  5.11s/it]

Found 647 point records in Mira, Veneto, Italy
Saved csv data to G:\CODE\data\05_landuse_mix\landuse_data.csv



C:\Users\ASUS\AppData\Local\Temp\ipykernel_18868\1417411790.py:57: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  final_gdf_points.to_file(shpfilename)
g:\CODE\0518\bear01\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'caravan:rental' to 'caravan_re'
  ogr_write(
g:\CODE\0518\bear01\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
g:\CODE\0518\bear01\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
g:\CODE\0518\bear01\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:postcode' to 'addr_postc'
  ogr_write(
g:\CODE\0518\bear01\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:street' to 'addr_stree'
  ogr_write(
g:\CODE\0518\bear01\lib\site-packag

Saved shp data to G:\CODE\data\05_landuse_mix\landuse_data.shp
